#### **Data Quality Management & Governance Framework**

**Objective**: Implement enterprise-grade data quality checks, validation rules, exception logging, and remediation tracking

#### Connect to MySQL

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import mysql.connector
from datetime import datetime

load_dotenv()

DB_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': 'youth_employment_db'
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Connected to MySQL")

Connected to MySQL


#### Create Data Quality Tables

In [2]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dq_rules (
    rule_id INT AUTO_INCREMENT PRIMARY KEY,
    rule_name VARCHAR(100),
    table_name VARCHAR(50),
    column_name VARCHAR(50),
    rule_type VARCHAR(50),          
    threshold FLOAT,
    description TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dq_exceptions (
    exception_id INT AUTO_INCREMENT PRIMARY KEY,
    participant_id VARCHAR(20),
    table_name VARCHAR(50),
    column_name VARCHAR(50),
    rule_name VARCHAR(100),
    issue_description TEXT,
    severity VARCHAR(20),           
    status VARCHAR(20) DEFAULT 'Open',  
    reported_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    resolved_date TIMESTAMP NULL,
    assigned_to VARCHAR(100)
)
""")

print("Data Quality governance tables created")

Data Quality governance tables created


#### Define & Insert DQ Rules

In [3]:
dq_rules = [
    ('completeness_participant_id', 'fact_program_performance', 'participant_id', 'completeness', 1.0, 'Participant ID must not be null'),
    ('valid_gender', 'fact_program_performance', 'gender', 'validity', 0.0, 'Gender must be Male or Female'),
    ('valid_placed', 'fact_program_performance', 'placed', 'validity', 0.0, 'Placed flag must be 0 or 1'),
    ('reasonable_income', 'fact_program_performance', 'monthly_income_usd', 'range', 0.0, 'Monthly income between 50 and 1500 USD'),
    ('valid_retention', 'fact_program_performance', 'still_employed_6m', 'validity', 0.0, 'Retention flag must be 0 or 1'),
]

cursor.executemany("""
    INSERT IGNORE INTO dq_rules (rule_name, table_name, column_name, rule_type, threshold, description)
    VALUES (%s, %s, %s, %s, %s, %s)
""", dq_rules)

conn.commit()
print("DQ Rules registered")

DQ Rules registered


#### Run Data Quality Checks & Log Exceptions

In [4]:
checks = []

# 1. Missing Income
cursor.execute("""
INSERT INTO dq_exceptions (participant_id, table_name, column_name, rule_name, issue_description, severity)
SELECT participant_id, 'fact_program_performance', 'monthly_income_usd', 'reasonable_income', 
       'Missing or null monthly income for placed participant', 'High'
FROM fact_program_performance 
WHERE placed = 1 AND (monthly_income_usd IS NULL OR monthly_income_usd < 50);
""")

# 2. Invalid Retention Flag
cursor.execute("""
INSERT INTO dq_exceptions (participant_id, table_name, column_name, rule_name, issue_description, severity)
SELECT participant_id, 'fact_program_performance', 'still_employed_6m', 'valid_retention',
       'Invalid retention value (not 0 or 1)', 'Medium'
FROM fact_program_performance 
WHERE still_employed_6m NOT IN (0,1);
""")

conn.commit()
print("Data quality checks executed and exceptions logged")

Data quality checks executed and exceptions logged


#### DQ Dashboard Query

In [5]:
dq_summary = pd.read_sql("""
    SELECT 
        rule_name,
        severity,
        COUNT(*) as exception_count,
        COUNT(CASE WHEN status = 'Resolved' THEN 1 END) as resolved
    FROM dq_exceptions 
    GROUP BY rule_name, severity
""", conn)

print("=== Data Quality Summary ===")
print(dq_summary)

# Overall DQ Score (Simple)
total_exceptions = pd.read_sql("SELECT COUNT(*) FROM dq_exceptions", conn).iloc[0,0]
print(f"\nTotal Open Exceptions: {total_exceptions}")

=== Data Quality Summary ===
         rule_name severity  exception_count  resolved
0  valid_retention   Medium              226         0

Total Open Exceptions: 226


C:\Users\tohiba\AppData\Local\Temp\ipykernel_10776\2009292168.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dq_summary = pd.read_sql("""
C:\Users\tohiba\AppData\Local\Temp\ipykernel_10776\2009292168.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_exceptions = pd.read_sql("SELECT COUNT(*) FROM dq_exceptions", conn).iloc[0,0]


#### Save & Close

In [6]:
conn.close()
print("Data Quality Framework ready")

Data Quality Framework ready
